In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]


In [15]:
from pymongo import MongoClient

usuario = "lipemanoel3_db_user"
senha = "8X0Um1y2V1ovJV5y"
servidor = "ibd016.tx9xomk.mongodb.net"

uri = "mongodb+srv://" + usuario + ":" + senha + "@" + servidor + "/?retryWrites=true&w=majority"

client = MongoClient(uri)

print(client.admin.command("ping"))

{'ok': 1}


In [16]:
db = client["ibd016_aula9"]

produtos = db["produtos"]

produtos.delete_many({})

DeleteResult({'n': 0, 'electionId': ObjectId('7fffffff0000000000000083'), 'opTime': {'ts': Timestamp(1793412384, 4), 't': 131}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1793412384, 4), 'signature': {'hash': b'\xa3\xf9\x00\xf660|r\x8a\x92\xe3\x1c\x0b\x08\x10PC=Q<', 'keyId': 7649792965602181126}}, 'operationTime': Timestamp(1793412384, 4)}, acknowledged=True)

In [17]:
doc = {
    "nome": "Teclado Mecânico",
    "preco": 350.0,
    "categoria": "periféricos",
    "estoque": 42,
    "tags": ["gamer", "usb"],
    "especificacoes": {
        "switch": "red",
        "layout": "ABNT2",
        "retroiluminado": True
    }
}

resultado = produtos.insert_one(doc)

print("Inserido com _id:", resultado.inserted_id)

Inserido com _id: 6ae54d363e7647038f239e7a


In [18]:
lote = [
    {"nome": "Mouse Sem Fio", "preco": 120.0, "categoria": "periféricos", "estoque": 80, "tags": ["sem fio", "usb"]},
    {"nome": "Monitor 24 polegadas", "preco": 899.9, "categoria": "monitores", "estoque": 15,
     "especificacoes": {"resolucao": "1920x1080", "painel": "IPS", "hz": 75}},
    {"nome": "Headset Gamer", "preco": 249.0, "categoria": "áudio", "estoque": 30, "tags": ["gamer", "usb", "microfone"]},
    {"nome": "Webcam Full HD", "preco": 199.0, "categoria": "periféricos", "estoque": 0, "tags": ["usb", "reunião"]},
    {"nome": "Cadeira Ergonômica", "preco": 1250.0, "categoria": "mobiliário", "estoque": 6, "garantia_meses": 24},
]

res = produtos.insert_many(lote)

print(len(res.inserted_ids), "documentos inseridos")
print("Total na coleção:", produtos.count_documents({}))

5 documentos inseridos
Total na coleção: 6


In [19]:
for p in produtos.find({"categoria": "periféricos"}):
    print(p["nome"], p["preco"])

webcam = produtos.find_one({"nome": "Webcam Full HD"})
print(webcam)

Teclado Mecânico 350.0
Mouse Sem Fio 120.0
Webcam Full HD 199.0
{'_id': ObjectId('6ae54d633e7647038f239e7e'), 'nome': 'Webcam Full HD', 'preco': 199.0, 'categoria': 'periféricos', 'estoque': 0, 'tags': ['usb', 'reunião']}


In [20]:
for p in produtos.find({"preco": {"$gt": 200}}, {"nome": 1, "preco": 1, "_id": 0}).sort("preco", -1):
    print(p)

{'nome': 'Cadeira Ergonômica', 'preco': 1250.0}
{'nome': 'Monitor 24 polegadas', 'preco': 899.9}
{'nome': 'Teclado Mecânico', 'preco': 350.0}
{'nome': 'Headset Gamer', 'preco': 249.0}


In [21]:
print("Produtos com a tag gamer:")
for p in produtos.find({"tags": "gamer"}):
    print(p["nome"])

print("Produtos com painel IPS:")
for p in produtos.find({"especificacoes.painel": "IPS"}):
    print(p["nome"])

Produtos com a tag gamer:
Teclado Mecânico
Headset Gamer
Produtos com painel IPS:
Monitor 24 polegadas


In [22]:
print("Sem estoque ou com preço abaixo de 130:")
for p in produtos.find({"$or": [{"estoque": 0}, {"preco": {"$lt": 130}}]}):
    print(p["nome"])

print("Categoria áudio ou monitores:")
for p in produtos.find({"categoria": {"$in": ["áudio", "monitores"]}}):
    print(p["nome"])

Sem estoque ou com preço abaixo de 130:
Mouse Sem Fio
Webcam Full HD
Categoria áudio ou monitores:
Monitor 24 polegadas
Headset Gamer


In [23]:
print("Com estoque:", produtos.count_documents({"estoque": {"$gt": 0}}))

Com estoque: 5


In [24]:
# 5.1 — $set: reajustar o preço de um produto
r = produtos.update_one({"nome": "Mouse Sem Fio"}, {"$set": {"preco": 129.9}})
print("encontrados:", r.matched_count, "| alterados:", r.modified_count)

# 5.2 — $inc: baixa de estoque após uma venda
produtos.update_one({"nome": "Teclado Mecânico"}, {"$inc": {"estoque": -1}})

# 5.3 — $push: acrescentar um item a uma lista
produtos.update_one({"nome": "Teclado Mecânico"}, {"$push": {"tags": "mecânico"}})

# 5.4 — update_many: alterar vários documentos de uma vez
produtos.update_many({"estoque": 0}, {"$set": {"disponivel": False}})
produtos.update_many({"estoque": {"$gt": 0}}, {"$set": {"disponivel": True}})

teclado = produtos.find_one({"nome": "Teclado Mecânico"})
print(teclado)

encontrados: 1 | alterados: 1
{'_id': ObjectId('6ae54d363e7647038f239e7a'), 'nome': 'Teclado Mecânico', 'preco': 350.0, 'categoria': 'periféricos', 'estoque': 41, 'tags': ['gamer', 'usb', 'mecânico'], 'especificacoes': {'switch': 'red', 'layout': 'ABNT2', 'retroiluminado': True}, 'disponivel': True}


In [25]:
r = produtos.delete_one({"nome": "Cadeira Ergonômica"})
print("removidos:", r.deleted_count)

r = produtos.delete_many({"disponivel": False})
print("removidos:", r.deleted_count)

print("Restam:", produtos.count_documents({}))

removidos: 1
removidos: 1
Restam: 4


In [26]:
produtos.create_index("categoria")
print(produtos.index_information())

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'categoria_1': {'v': 2, 'key': [('categoria', 1)]}}


In [27]:
tarefas = db["tarefas"]

tarefas.delete_many({})

lista_tarefas = [
    {
        "titulo": "Estudar MongoDB",
        "prioridade": "alta",
        "responsavel": "Lipe",
        "tags": ["estudo", "banco de dados"],
        "concluida": False,
        "criada_em": "2026-10-24"
    },
    {
        "titulo": "Fazer trabalho da faculdade",
        "prioridade": "media",
        "responsavel": "João",
        "tags": ["faculdade", "trabalho"],
        "concluida": False,
        "criada_em": "2026-10-24"
    },
    {
        "titulo": "Revisar SQL",
        "prioridade": "alta",
        "responsavel": "Lipe",
        "tags": ["estudo", "SQL"],
        "concluida": True,
        "criada_em": "2026-10-24",
        "observacao": "Revisar comandos SELECT"
    },
    {
        "titulo": "Entregar atividade",
        "prioridade": "baixa",
        "responsavel": "Maria",
        "tags": ["faculdade"],
        "concluida": False,
        "criada_em": "2026-10-24"
    }
]

tarefas.insert_many(lista_tarefas)

print("Total de tarefas:", tarefas.count_documents({}))

Total de tarefas: 4


In [28]:
for t in tarefas.find({
    "prioridade": "alta",
    "concluida": False
}):
    print(t)

{'_id': ObjectId('6ae54f963e7647038f239e80'), 'titulo': 'Estudar MongoDB', 'prioridade': 'alta', 'responsavel': 'Lipe', 'tags': ['estudo', 'banco de dados'], 'concluida': False, 'criada_em': '2026-10-24'}


In [29]:
for t in tarefas.find({"tags": "estudo"}):
    print(t["titulo"])

Estudar MongoDB
Revisar SQL


In [30]:
for t in tarefas.find(
    {},
    {"titulo": 1, "responsavel": 1, "_id": 0}
).sort("titulo", 1):
    print(t)

{'titulo': 'Entregar atividade', 'responsavel': 'Maria'}
{'titulo': 'Estudar MongoDB', 'responsavel': 'Lipe'}
{'titulo': 'Fazer trabalho da faculdade', 'responsavel': 'João'}
{'titulo': 'Revisar SQL', 'responsavel': 'Lipe'}


In [31]:
print("Tarefas concluídas:", tarefas.count_documents({
    "concluida": True
}))

Tarefas concluídas: 1


In [32]:
tarefas.update_one(
    {"titulo": "Estudar MongoDB"},
    {"$set": {"concluida": True}}
)

print("Tarefa atualizada!")

Tarefa atualizada!


In [33]:
tarefas.update_one(
    {"titulo": "Fazer trabalho da faculdade"},
    {"$addToSet": {"tags": "urgente"}}
)

print("Tag adicionada!")

Tag adicionada!


In [34]:
tarefas.update_many(
    {"responsavel": "Lipe"},
    {"$set": {"prioridade": "media"}}
)

print("Prioridades atualizadas!")

Prioridades atualizadas!


In [35]:
for t in tarefas.find():
    print(t)

{'_id': ObjectId('6ae54f963e7647038f239e80'), 'titulo': 'Estudar MongoDB', 'prioridade': 'media', 'responsavel': 'Lipe', 'tags': ['estudo', 'banco de dados'], 'concluida': True, 'criada_em': '2026-10-24'}
{'_id': ObjectId('6ae54f963e7647038f239e81'), 'titulo': 'Fazer trabalho da faculdade', 'prioridade': 'media', 'responsavel': 'João', 'tags': ['faculdade', 'trabalho', 'urgente'], 'concluida': False, 'criada_em': '2026-10-24'}
{'_id': ObjectId('6ae54f963e7647038f239e82'), 'titulo': 'Revisar SQL', 'prioridade': 'media', 'responsavel': 'Lipe', 'tags': ['estudo', 'SQL'], 'concluida': True, 'criada_em': '2026-10-24', 'observacao': 'Revisar comandos SELECT'}
{'_id': ObjectId('6ae54f963e7647038f239e83'), 'titulo': 'Entregar atividade', 'prioridade': 'baixa', 'responsavel': 'Maria', 'tags': ['faculdade'], 'concluida': False, 'criada_em': '2026-10-24'}


In [36]:
resultado = tarefas.delete_many({"concluida": True})

print("Tarefas removidas:", resultado.deleted_count)
print("Tarefas restantes:", tarefas.count_documents({}))

Tarefas removidas: 2
Tarefas restantes: 2


In [37]:
tarefas.create_index("prioridade")

print(tarefas.index_information())

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'prioridade_1': {'v': 2, 'key': [('prioridade', 1)]}}
